In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.runnables import RunnableLambda, RunnableBranch

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

In [24]:
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie review evaluator."),
        ("user", "Please categorize the movie review as positive or negative: {input}")
    ]
)
my_template

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a movie review evaluator.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='Please categorize the movie review as positive or negative: {input}'), additional_kwargs={})])

In [25]:
llm_structured_output = llm.with_structured_output(llm_schema)

In [31]:
#### Convert pydantic object into a dictionary

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)

##### Based on the feedback, chain will be executed

#### Chain 1

In [12]:
#### Task 1: Prompt
prompt_post = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a LinkedIn post generator."),
        ("user", "Create a post for the following text for LinkedIn: {text}")
    ]
)

# Task 2: LLM
llm_generate = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Task 3: Parser
str_parser = StrOutputParser()

chain_linkedin = prompt_post | llm_generate | str_parser

#### Chain 2

In [13]:
def insta_chain(text : dict): 
    
    text = text["text"]
    
    #### Task 1: Prompt
    prompt_post = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a Instagram post generator."),
            ("user", "Create a post for the following text for Instagram: {text}")
        ]
    )

    # Task 2: LLM
    llm_generate = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    # Task 3: Parser
    str_parser = StrOutputParser()
    
    chain_insta = prompt_post | llm_generate | str_parser
    response = chain_insta.invoke(text)
    return response


insta_chain_runnable = RunnableLambda(insta_chain)

#### Final Orchestration

In [33]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable ## default
)

final_orchestrator = my_template | llm_structured_output | pydantic_json_runnable | conditional_chain
final_orchestrator.invoke({"input": "I loved this KFG movie!"})

'Okay, here are a few options for a LinkedIn post based on the word "positive," each with a slightly different angle. Choose the one that best fits your personal brand or the message you want to convey!\n\n---\n\n**Option 1: General Mindset & Impact**\n\n✨ Starting the week with a powerful reminder: the ripple effect of a #PositiveMindset.\n\nIt\'s not about ignoring challenges, but choosing how we approach them. A positive outlook can transform obstacles into opportunities, fuel creativity, and inspire those around us. Let\'s consciously choose optimism today.\n\nWhat\'s one positive habit you practice to keep your energy high and your focus sharp? Share below! 👇\n\n#PositiveVibes #MindsetMatters #WorkLife #Motivation\n\n---\n\n**Option 2: Workplace Culture & Teamwork**\n\nIn every team, in every project, and in every interaction, the power of a #Positive attitude is undeniable. 🚀\n\nIt fosters collaboration, encourages open communication, and turns "we can\'t" into "how can we?" Let\